CSCI580 Final Project Group 9

Imports
Source:
- Final Project Slides: project requires PyTorch, MNIST, custom PNG data, accuracy statistics, F1-score, and confusion matrix.
- PerfEval slides: model performance metrics.
- Files: BuildNN-1.ipynb and LoadImage.ipynb

In [ ]:
import os                           # For file/directory handling
import random                       # For reproducibility
from collections import defaultdict # For grouping

import numpy as np                  # Numerical operations
import matplotlib.pyplot as plt     # Plotting graphs
from IPython.display import display

from PIL import Image               # Image loading

import torch                        # Core PyTorch library
import torch.nn as nn               # Neural network modules
import torch.optim as optim         # Optimizers

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms  # Datasets and preprocessing

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

Reproducibility + Device Setup
Source:
- General PyTorch setup

In [ ]:
# Locking seed to group number for consistent data
seed = 9

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Image Transform / Preprocessing
Source:
- Final Project slides, Task 3:
  ToTensor() maps [0,255] -> [0,1]
  Normalize(mean=0.5, std=0.5) maps [0,1] -> [-1,1]

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

Load MNIST Dataset
Source:
- Final Project slides, page 8:
  trainset = datasets.MNIST(... train=True)
  testset = datasets.MNIST(... train=False)

In [ ]:
mnist_train_full = datasets.MNIST(
    root="./MNIST_data",
    train=True,
    download=True,
    transform=transform
)

mnist_test = datasets.MNIST(
    root="./MNIST_data",
    train=False,
    download=True,
    transform=transform
)


Source:
- Model Tuning slides, slide "Training, Validation and Test"
- Model Tuning slides, slide "Training with Validation"


In [ ]:
train_size = int(0.8 * len(mnist_train_full))
val_size = len(mnist_train_full) - train_size

mnist_train, mnist_val = random_split(
    mnist_train_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(seed)
)

batch_size = 64

train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(mnist_val, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

print("Training samples:", len(mnist_train))
print("Validation samples:", len(mnist_val))
print("MNIST test samples:", len(mnist_test))

Custom Project Dataset Loader
Source:
- Final Project slides, Task 1:
  filename format: <label/digit>-<groupID>-<memberID>.png
- Final Project slides, Task 2:
  read PNG files and assign labels
- Final Project slides, Task 3:
  preprocess images using PyTorch transforms

In [ ]:
class ProjectDigitDataset(Dataset):
    def __init__(self, folder_path):
        self.folder_path = folder_path

        self.files = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith(".png")
        ]

        self.files.sort()

        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]

        # Expected filename:
        # <label/digit>-<groupID>-<memberID>.png
        # Example:
        # 3-4-1.png means digit 3, group 4, member 1

        name_without_extension = filename.replace(".png", "")
        parts = name_without_extension.split("-")

        label = int(parts[0])
        group_id = int(parts[1])
        member_id = int(parts[2])

        image_path = os.path.join(self.folder_path, filename)

        image = Image.open(image_path)
        image = self.transform(image)

        return image, label, group_id, member_id, filename


# Change this path if your custom digit folder is elsewhere
custom_digits_folder = "./digits"

if os.path.exists(custom_digits_folder):
    custom_dataset = ProjectDigitDataset(custom_digits_folder)
    custom_loader = DataLoader(custom_dataset, batch_size=32, shuffle=False)
    print("Custom digit samples:", len(custom_dataset))
else:
    custom_dataset = None
    custom_loader = None
    print("No custom digits folder found yet.")

MLP Model Definition
Source:
- Final Project slides:
  Must use MLP / fully connected feedforward neural network only.
- BuildTrainNN slides:
  Example architecture uses input 784, hidden layers, output 10.
- Model Tuning slides:
  Architecture hyperparameters include number of layers,
  neurons per layer, activation function, and dropout.

In [ ]:
class MLP(nn.Module):
    def __init__(self, network, dropout_rate=0.3):
        super().__init__()

        self.network = network

    def forward(self, x):
        # Flatten 28x28 image into 784 input features.
        x = x.view(x.shape[0], -1)
        return self.network(x)

Count Parameters
Source:
- Final Report requirements:
  include number of neurons, layers, weights, and key hyperparameters.

In [ ]:
def count_parameters(model):
    total = 0
    trainable = 0

    for param in model.parameters():
        num = param.numel()
        total += num

        if param.requires_grad:
            trainable += num

    return total, trainable


Training Function
Source:
- BuildTrainNN slides, training main loop:
  - optimizer.zero_grad()
  - output = model(input)
  - loss = loss_fn(output, target)
  - loss.backward()
  - optimizer.step()

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()
        # Update weights
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

Validation Function

Source:
- Model Tuning slides, slide "Training with Validation":
  - validation phase after each epoch
  - model.eval()
  - disable gradient computation
  - forward pass only
  - compute validation loss and metrics

In [ ]:
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # Disable gradients
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

Train Model Across Epochs
Source:
- Model Tuning slides:
  track training and validation loss.
- Final Project slides:
  validate while training and tune hyperparameters.

In [ ]:
def train_model(model, criterion, optimizer, epochs):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_loss = float("inf")
    best_model_path = "best_mlp_model.pth"
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )
    
        val_loss, val_acc = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )
    
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)
    
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
    
        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_acc:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Acc: {val_acc:.4f}"
        )
    
    model.load_state_dict(torch.load(best_model_path, map_location=device))

    figures = []
    
    # plt.figure()
    fig, ax = plt.subplots()
    plt.plot(range(1, epochs + 1), train_losses, label="Training Loss")
    plt.plot(range(1, epochs + 1), val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    figures.append(fig)
    plt.close()
    
    # plt.figure()
    fig, ax = plt.subplots()
    plt.plot(range(1, epochs + 1), train_accuracies, label="Training Accuracy")
    plt.plot(range(1, epochs + 1), val_accuracies, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    figures.append(fig)    
    plt.close()

    return figures

Plot Training vs Validation Loss
Source:
- Model Tuning slides:
  overfitting slides show training vs validation loss.
- Final Project slides:
  report requires training vs validation loss over iterations.

General Evaluation Function for MNIST Test Data
Source:
- Final Project slides:
  test on MNIST test images.
- PerfEval slides:
  accuracy, error rate, F1-score, and confusion matrix.

In [ ]:
def evaluate_mnist_test_data(model, dataloader, device, dataset_name="MNIST Test Data"):
    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    error_rate = 1 - accuracy

    f1_per_digit = f1_score(
        all_labels,
        all_predictions,
        average=None,
        labels=list(range(10)),
        zero_division=0
    )

    cm = confusion_matrix(
        all_labels,
        all_predictions,
        labels=list(range(10))
    )

    print(dataset_name)
    print("Accuracy:", accuracy)
    print("Error Rate:", error_rate)

    print("\nF1 Score Per Digit:")
    for digit, score in enumerate(f1_per_digit):
        print(f"Digit {digit}: {score:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            all_labels,
            all_predictions,
            labels=list(range(10)),
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "error_rate": error_rate,
        "f1_per_digit": f1_per_digit,
        "confusion_matrix": cm,
        "labels": all_labels,
        "predictions": all_predictions
    }

Custom Image Evaluation Per Group
Source:
- Final Project slides:
  custom filename format:
  <label/digit>-<groupID>-<memberID>.png
- Final Project slides:
  final evaluation should include each group's images.
- PerfEval slides:
  accuracy, error rate, F1-score, confusion matrix.

In [ ]:
def evaluate_custom_images_by_group(model, dataloader, device):
    model.eval()

    group_results = defaultdict(lambda: {
        "labels": [],
        "predictions": [],
        "filenames": []
    })

    with torch.no_grad():
        for images, labels, group_ids, member_ids, filenames in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            labels_np = labels.cpu().numpy()
            predictions_np = predictions.cpu().numpy()
            group_ids_np = group_ids.numpy()

            for label, pred, group_id, filename in zip(
                labels_np,
                predictions_np,
                group_ids_np,
                filenames
            ):
                group_id = int(group_id)

                group_results[group_id]["labels"].append(int(label))
                group_results[group_id]["predictions"].append(int(pred))
                group_results[group_id]["filenames"].append(filename)

    print("Group Collected Digit Images - Per Group")

    final_results = {}

    for group_id in sorted(group_results.keys()):
        labels = group_results[group_id]["labels"]
        predictions = group_results[group_id]["predictions"]

        accuracy = accuracy_score(labels, predictions)
        error_rate = 1 - accuracy

        macro_f1 = f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )

        cm = confusion_matrix(
            labels,
            predictions,
            labels=list(range(10))
        )

        print(f"\nGroup {group_id}")
        print("------------------------------")
        print("Number of images:", len(labels))
        print("Accuracy:", accuracy)
        print("Error Rate:", error_rate)
        print("Macro F1 Score:", macro_f1)

        final_results[group_id] = {
            "accuracy": accuracy,
            "error_rate": error_rate,
            "macro_f1": macro_f1,
            "confusion_matrix": cm,
            "labels": labels,
            "predictions": predictions,
            "filenames": group_results[group_id]["filenames"]
        }

    return final_results

Plot Confusion Matrix
Source:
- PerfEval slides:
  confusion matrix summarizes classification performance.
- Matplotlib documentation:
  https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html
- Geeksforgeeks:
  https://www.geeksforgeeks.org/python/matplotlib-pyplot-imshow-in-python/

In [ ]:
def plot_confusion_matrix(cm, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    fig.colorbar(im, ax=ax)

    tick_marks = np.arange(10)
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(range(10))
    ax.set_yticklabels(range(10))

    for i in range(10):
        for j in range(10):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    fig.tight_layout()
    plt.close(fig)
    return fig

In [ ]:
def try_model(network, dropout_rate, learning_rate, epochs):
    model = MLP(network, dropout_rate=0.3).to(device)
    total_params, trainable_params = count_parameters(model)
    
    # Display parameters
    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    
    criterion = nn.CrossEntropyLoss()
        
    # Loss Function and Optimizer
    # Source:
    # - BuildTrainNN slides:
    #   discusses NLLLoss, CrossEntropyLoss, SGD, Adam.
    # - PerfEval slides:
    #   CrossEntropyLoss / log loss is used during training.
    # - Model Tuning slides:
    #   optimizer type, learning rate, and weight decay are hyperparameters.
    
    # Initialize ADAM optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )

    figs = train_model(model, criterion, optimizer, epochs)

    return model, figs


Evaluate on MNIST Test Data
    Source:
    - Final Project slides:
      test handwritten digit recognition using MNIST test data.

In [ ]:
def display_mnist_results(model):
    return evaluate_mnist_test_data(
        model,
        test_loader,
        device,
        dataset_name="MNIST Test Data"
    )

Evaluate on Group-Collected Custom Images
Source:
- Final Project slides:
  test handwritten digit recognition using all group-collected images.
  Accuracy evaluation should be per FinalProjectGroup.

In [ ]:
def display_custom_results(model):
    if custom_loader is not None:
        return evaluate_custom_images_by_group(
            model,
            custom_loader,
            device
        )
    else:
        custom_group_results = None
        print("Skipping custom image evaluation because ./digits was not found.")

In [ ]:
def display_confusion_matrix(mnist_results):
    return plot_confusion_matrix(
        mnist_results["confusion_matrix"],
        title="MNIST Test Confusion Matrix"
    )

Save Final Model
Source:
- Final Project slides:
  GitHub submission should include experiment materials,
  results, and trained model.

In [ ]:
def save_model(model, name):
    torch.save(model.state_dict(), name)
    print("Saved model as " + name)
    # torch.save(model.state_dict(), "final_mlp_digit_classifier.pth")
    # print("Saved model as final_mlp_digit_classifier.pth")

This block is an example of what we can copy paste and modify to test different stuff

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 256),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(128, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.01
epochs = 10
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 256),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(128, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 1024),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(1024, 256),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 10
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 1024),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(1024, 256),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
save_model(model, "large_precise_model.pth")

In [ ]:
custom_results = display_custom_results(model)

In [ ]:
display_confusion_matrix(mnist_results)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.005
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 1024),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(1024, 256),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 256),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 128),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(128, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 128),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(128, 32),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(32, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 32),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(32, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 32),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(32, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.001
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 5024),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(5024, 256),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
dropout_rate = 0.3
learning_rate = 0.005
epochs = 3
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 10),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(10, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
# model = MLP(nn.Sequential(
#                 nn.Linear(784, 1024),
#                 nn.ReLU(),
#                 nn.Dropout(dropout_rate),
    
#                 nn.Linear(1024, 256),
#                 nn.ReLU(),
#                 nn.Dropout(dropout_rate),
    
#                 nn.Linear(256, 10)
#             ), dropout_rate=0.3).to(device)

# state_dict = torch.load('large_precise_model.pth', weights_only=True)

# model.load_state_dict(state_dict)
# model.eval()

# mnist_results = display_mnist_results(model)
# custom_results = display_custom_results(model)

In [ ]:
dropout_rate = 0.5
learning_rate = 0.001
epochs = 10
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 1024),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(1024, 256),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
custom_results = display_custom_results(model)

In [ ]:
dropout_rate = 0.1
learning_rate = 0.001
epochs = 10
model, figs = try_model(nn.Sequential(
                nn.Linear(784, 1024),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(1024, 256),
                nn.Sigmoid(),
                nn.Dropout(dropout_rate),
    
                nn.Linear(256, 10)
            ),
         dropout_rate,
         learning_rate,
         epochs)

for fig in figs:
    display(fig)

mnist_results = display_mnist_results(model)

In [ ]:
custom_results = display_custom_results(model)